#**Local Stochastic Jump-Volatility (LSJV) with AAD**

Ganett Bertrand Pardomuan Hutahaean

This notebook isolates the Heston stochastic volatility dynamics, the Merton jump compensators, and validates the Log-Euler time-stepping scheme.

## 1. The Core Dynamics (Bates Model SDE)
The asset price $S_t$ and variance $v_t$ follow:

$$dS_t = (r - q - \lambda \bar{\mu}) S_t dt + \sqrt{v_t} S_t dW_t^{(1)} + S_{t-}(e^J - 1) dN_t$$
$$dv_t = \kappa(\theta - v_t) dt + \xi \sqrt{v_t} dW_t^{(2)}$$

*(Note: The Local Volatility leverage function $L(S_t, t)$ is omitted here to isolate the pure parametric jump-diffusion).*

## 2. Why Log-Euler?
Standard Euler-Maruyama discretization can push $S_t$ or $v_t$ below zero during large simulated steps. We use a full-truncation scheme for variance ($v^+ = \max(v, 0)$) and a Log-Euler scheme for the asset price to guarantee strict positivity:
$$S_{t+\Delta t} = S_t \exp\left( (r - q - \lambda \bar{\mu} - 0.5 v_t) \Delta t + \sqrt{v_t \Delta t} Z_1 \right) \cdot \exp(J \cdot \Delta N_t)$$

##Environment Setup

In [20]:
import torch
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

torch.set_default_dtype(torch.float64)

##Isolating the Continuous Diffusion (Heston)

In [21]:
# Simulate Pure Heston Dynamics (No Jumps)
# Test the correlation (rho) between asset price and variance.

def simulate_heston(S0=100, v0=0.04, kappa=2.0, theta=0.04, xi=0.5, rho=-0.7, T=1.0, steps=252):
    dt = T / steps

    # Generate correlated Brownian motions
    Z1 = torch.randn(steps)
    Z2 = torch.randn(steps)
    W_v = Z1
    W_S = rho * Z1 + torch.sqrt(torch.tensor(1.0 - rho**2)) * Z2

    S_path = [torch.tensor(S0, dtype=torch.float64)]
    v_path = [torch.tensor(v0, dtype=torch.float64)]

    S, v = S_path[0], v_path[0]

    for i in range(steps):
        v_pos = torch.clamp(v, min=0.0)

        # Variance step (Euler)
        v_next = v + kappa * (theta - v_pos) * dt + xi * torch.sqrt(v_pos) * torch.sqrt(torch.tensor(dt)) * W_v[i]

        # Asset step (Log-Euler)
        drift = -0.5 * v_pos * dt
        diffusion = torch.sqrt(v_pos) * torch.sqrt(torch.tensor(dt)) * W_S[i]
        S_next = S * torch.exp(drift + diffusion)

        S, v = S_next, v_next
        S_path.append(S)
        v_path.append(v)

    return torch.stack(S_path), torch.stack(v_path)

S_heston, v_heston = simulate_heston()
print("Heston continuous paths generated successfully.")

Heston continuous paths generated successfully.


##Isolating the Jump Component (Merton)

In [22]:
# Simulate the Poisson Jumps
# Isolate the jump compensator to ensure the process remains a martingale.

def simulate_jumps(lambda_jump=3.0, mu_J=-0.1, sig_J=0.15, T=1.0, steps=252):
    dt = T / steps

    # Compensator: \bar{\mu} = E[e^J - 1]
    compensator = np.exp(mu_J + 0.5 * sig_J**2) - 1.0

    # Poisson process for jump occurrences
    poisson_dist = torch.distributions.Poisson(lambda_jump * dt)
    dN = poisson_dist.sample((steps,))

    # Normal distribution for jump magnitudes
    J = torch.randn(steps) * sig_J + mu_J

    # Jump multiplier: e^(J * dN)
    jump_multipliers = torch.exp(J * dN)

    # Cumulative jump effect on an asset starting at 100 (compensated)
    S_jump_only = [torch.tensor(100.0, dtype=torch.float64)]
    S = S_jump_only[0]

    for i in range(steps):
        # Apply drift compensator and the actual jump (if any)
        drift = (-lambda_jump * compensator) * dt
        S_next = S * torch.exp(torch.tensor(drift)) * jump_multipliers[i]

        S = S_next
        S_jump_only.append(S)

    return torch.stack(S_jump_only), dN, J

S_jumps, dN, J = simulate_jumps()
print(f"Total jumps simulated in 1 year: {dN.sum().item()}")

Total jumps simulated in 1 year: 3.0


##Combining into the Full Bates Model SDE

In [23]:
# The Full Bates Model (Heston + Merton Jumps)

def simulate_bates(S0=100, v0=0.04, r=0.05,
                   kappa=2.0, theta=0.04, xi=0.5, rho=-0.7,
                   lambda_jump=3.0, mu_J=-0.1, sig_J=0.15,
                   T=1.0, steps=252):

    dt = T / steps
    compensator = np.exp(mu_J + 0.5 * sig_J**2) - 1.0

    Z1 = torch.randn(steps)
    Z2 = torch.randn(steps)
    W_v = Z1
    W_S = rho * Z1 + torch.sqrt(torch.tensor(1.0 - rho**2)) * Z2

    poisson_dist = torch.distributions.Poisson(lambda_jump * dt)
    dN = poisson_dist.sample((steps,))
    J = torch.randn(steps) * sig_J + mu_J
    jump_multipliers = torch.exp(J * dN)

    S_path = [torch.tensor(S0, dtype=torch.float64)]
    v_path = [torch.tensor(v0, dtype=torch.float64)]
    S, v = S_path[0], v_path[0]

    for i in range(steps):
        v_pos = torch.clamp(v, min=0.0)

        v_next = v + kappa * (theta - v_pos) * dt + xi * torch.sqrt(v_pos) * torch.sqrt(torch.tensor(dt)) * W_v[i]

        drift = (r - lambda_jump * compensator - 0.5 * v_pos) * dt
        diffusion = torch.sqrt(v_pos) * torch.sqrt(torch.tensor(dt)) * W_S[i]
        S_next = S * torch.exp(drift + diffusion) * jump_multipliers[i]

        S, v = S_next, v_next
        S_path.append(S)
        v_path.append(v)

    return torch.stack(S_path), torch.stack(v_path), dN

S_bates, v_bates, jump_flags = simulate_bates()
print("Full Bates model path simulated successfully.")

Full Bates model path simulated successfully.


##Visualizing the Mechanics

In [26]:
# Visualizing the SDE Components
# We plot the isolated components vs the combined model to understand the dynamics.

time_grid = np.linspace(0, 1.0, 253)

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.08,
    subplot_titles=("Asset Price Dynamics", "Variance Process (Heston)", "Isolated Jumps (Poisson Process)")
)

# Plot 1: Asset Paths
fig.add_trace(go.Scatter(x=time_grid, y=S_heston.numpy(), mode='lines', name='Pure Heston (No Jumps)', line=dict(color='gray')), row=1, col=1)
fig.add_trace(go.Scatter(x=time_grid, y=S_bates.numpy(), mode='lines', name='Bates (Heston + Jumps)', line=dict(color='blue', width=2)), row=1, col=1)

# Highlight where jumps occurred on the Bates path
jump_indices = torch.where(jump_flags > 0)[0].numpy() + 1
# Use np.atleast_1d to ensure Plotly receives an array even for a single jump
fig.add_trace(go.Scatter(
    x=np.atleast_1d(time_grid[jump_indices]),
    y=np.atleast_1d(S_bates.numpy()[jump_indices]),
    mode='markers',
    name='Jump Events',
    marker=dict(color='red', size=8, symbol='x')
), row=1, col=1)

# Plot 2: Variance
fig.add_trace(go.Scatter(x=time_grid, y=v_bates.numpy(), mode='lines', name='Stochastic Variance (v_t)', line=dict(color='purple')), row=2, col=1)

# Plot 3: Isolated Jumps
fig.add_trace(go.Scatter(x=time_grid, y=S_jumps.numpy(), mode='lines', name='Compensated Jumps Only', line=dict(color='orange')), row=3, col=1)

fig.update_layout(height=800, title_text="SDE Component Breakdown: Building the Bates Model", template="plotly_white")
fig.update_yaxes(title_text="Asset Price", row=1, col=1)
fig.update_yaxes(title_text="Variance", row=2, col=1)
fig.update_yaxes(title_text="Isolated Price", row=3, col=1)

fig.show()

##Conclusion:
Notice how the pure Heston path (gray) is smooth and continuous, while the Bates path (blue) exhibits discrete gaps (marked with red 'X's). The variance process (purple) drives the "wiggle" of the path, while the isolated jumps (orange) prove our compensator correctly balances the negative mean drift.